# Habitat Risk — Existing Splits & Min–Max Normalization

This notebook rebuilds `data/processed/hri/model_v1/final_hri_model_dataset.csv` with sampled observation context using the existing builder, preserves its frozen rows and splits, and normalizes the nine continuous climate/terrain predictors using training statistics. It also scales four numeric context fields while preserving unavailable context as missing. It writes the existing `train.csv`, `validation.csv`, and `test.csv` into `data/splits/hri/model_v1/`. The observation source CSV stays unchanged.

`context__` columns summarize visited sites and are excluded from the existing model feature lists. Sampled NDVI is exploratory because its product/time-window/QA provenance is unresolved. Land-cover codes retain ESA/ESRI prefixes and are categorical. Unvisited cells and unknown survey effort remain missing; do not interpret them as ecological absence.

**Run all cells in order.** Cell 2 finds the component root when Jupyter starts from the repository root, component folder, `notebooks/`, or `notebooks/hri/`. Validation/test normalized values may be outside `[0,1]` when they exceed the training range; this is expected and avoids leakage.

## Cell 1 — Imports

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

## Cell 2 — Configure paths

In [18]:
# Find components/habitat_risk without depending on the Jupyter working directory.
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in search_roots if (p / "data/processed/hri/model_v1/final_hri_model_dataset.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (p / "components/habitat_risk" for p in search_roots
         if (p / "components/habitat_risk/data/processed/hri/model_v1/final_hri_model_dataset.csv").is_file()),
        None,
    )
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not find components/habitat_risk from: {cwd}")

DATA_PATH = PROJECT_ROOT / "data/processed/hri/model_v1/final_hri_model_dataset.csv"
SPLITS_DIR = PROJECT_ROOT / "data/splits/hri/model_v1"
NORMALIZE_COLS = [
    "climate__annual_mean_temperature_c",
    "climate__annual_precipitation_mm",
    "climate__temperature_c_anomaly",
    "climate__temperature_c_trailing5_slope_per_year",
    "climate__precipitation_mm_anomaly",
    "climate__precipitation_mm_trailing5_slope_per_year",
    "terrain__mean_elevation_m",
    "terrain__elevation_sd_m",
    "terrain__mean_slope_degrees",
]
print("Input:", DATA_PATH)
print("Output:", SPLITS_DIR)
assert DATA_PATH.is_file(), f"Dataset not found: {DATA_PATH}"

Input: /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/processed/hri/model_v1/final_hri_model_dataset.csv
Output: /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/splits/hri/model_v1


## Cell 3 — Load data and check existing split labels

In [19]:
# Rebuild the existing final dataset with observation context using the existing script.
import runpy
builder = runpy.run_path(str(PROJECT_ROOT / 'scripts/build_final_model_dataset.py'))
builder['main']()
df = pd.read_csv(DATA_PATH)
KEY = ["cell_id", "reference_year"]
TARGETS = ["hri_score_0_100", "fixed_risk_class", "quantile_risk_class"]
required = set(KEY + TARGETS + ["dataset_split"] + NORMALIZE_COLS)
missing_required = sorted(required - set(df.columns))
assert not missing_required, f"Required columns are missing: {missing_required}"
missing_cols = sorted(set(NORMALIZE_COLS) - set(df.columns))
assert not missing_cols, f"Missing normalization columns: {missing_cols}"
assert not df.duplicated(KEY).any(), "Duplicate cell_id + reference_year keys found."
assert df[TARGETS].notna().all().all(), "Targets must not contain missing values."
print("Shape:", df.shape)
print(df["dataset_split"].value_counts(dropna=False))

Shape: (35350, 24)
dataset_split
train         24752
validation     5306
test           5292
Name: count, dtype: int64


## Cell 4 — Use existing splits (do not regenerate)

In [20]:
# Preserve the existing split membership; do not generate or shuffle splits.
labels = df["dataset_split"].astype("string").str.strip().str.lower()
labels = labels.replace({"val": "validation", "valid": "validation", "training": "train", "testing": "test"})
assert labels.notna().all() and labels.isin(["train", "validation", "test"]).all(), "Unexpected or missing split labels."
df["dataset_split"] = labels
train_df = df.loc[labels == "train"].copy()
val_df = df.loc[labels == "validation"].copy()
test_df = df.loc[labels == "test"].copy()
assert all(len(d) > 0 for d in (train_df, val_df, test_df))
assert sum(map(len, (train_df, val_df, test_df))) == len(df)
assert df.groupby("cell_id")["dataset_split"].nunique().max() == 1, "A cell occurs in multiple splits."
split_cells = {name: set(part["cell_id"]) for name, part in (("train", train_df), ("validation", val_df), ("test", test_df))}
assert split_cells["train"].isdisjoint(split_cells["validation"])
assert split_cells["train"].isdisjoint(split_cells["test"])
assert split_cells["validation"].isdisjoint(split_cells["test"])
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})

{'train': 24752, 'validation': 5306, 'test': 5292}


## Cell 5 — Numeric checks and training-only median imputation

In [21]:
for part in (train_df, val_df, test_df):
    for col in NORMALIZE_COLS:
        part[col] = pd.to_numeric(part[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

assert train_df[NORMALIZE_COLS].notna().any().all(), "At least one normalization column has no valid training values."
imputer = SimpleImputer(strategy="median")
train_df[NORMALIZE_COLS] = imputer.fit_transform(train_df[NORMALIZE_COLS])
val_df[NORMALIZE_COLS] = imputer.transform(val_df[NORMALIZE_COLS])
test_df[NORMALIZE_COLS] = imputer.transform(test_df[NORMALIZE_COLS])
print("Training medians:", dict(zip(NORMALIZE_COLS, imputer.statistics_)))

Training medians: {'climate__annual_mean_temperature_c': np.float64(26.589855507100935), 'climate__annual_precipitation_mm': np.float64(1654.4846638532422), 'climate__temperature_c_anomaly': np.float64(0.1667184073719827), 'climate__temperature_c_trailing5_slope_per_year': np.float64(0.0022968858323537), 'climate__precipitation_mm_anomaly': np.float64(-358.8679021838892), 'climate__precipitation_mm_trailing5_slope_per_year': np.float64(-41.38124855753473), 'terrain__mean_elevation_m': np.float64(81.76356243406023), 'terrain__elevation_sd_m': np.float64(11.43945206981794), 'terrain__mean_slope_degrees': np.float64(1.8105955818928052)}


## Cell 6 — Training-only MinMaxScaler

In [22]:
scaler = MinMaxScaler()
train_df[NORMALIZE_COLS] = scaler.fit_transform(train_df[NORMALIZE_COLS])
val_df[NORMALIZE_COLS] = scaler.transform(val_df[NORMALIZE_COLS])
test_df[NORMALIZE_COLS] = scaler.transform(test_df[NORMALIZE_COLS])
print("Training min/max:")
print(train_df[NORMALIZE_COLS].agg(["min", "max"]))
print("Validation/test may exceed [0,1] if outside training range.")

# Optional context fields: scale available numeric measurements using training
# statistics; preserve missing evidence rather than inventing survey coverage.
CONTEXT_NORMALIZE_COLS = [
    'context__sampled_ndvi_mean',
    'context__sampled_elevation_mean_m',
    'context__survey_duration_mean_minutes',
    'context__survey_distance_mean_km',
]
assert train_df[CONTEXT_NORMALIZE_COLS].notna().any().all(), 'Context has no training measurements.'
context_scaler = MinMaxScaler()
train_df[CONTEXT_NORMALIZE_COLS] = context_scaler.fit_transform(train_df[CONTEXT_NORMALIZE_COLS])
val_df[CONTEXT_NORMALIZE_COLS] = context_scaler.transform(val_df[CONTEXT_NORMALIZE_COLS])
test_df[CONTEXT_NORMALIZE_COLS] = context_scaler.transform(test_df[CONTEXT_NORMALIZE_COLS])
print('Observation context scaled; categorical land cover and evidence counts remain unchanged.')

Training min/max:
     climate__annual_mean_temperature_c  climate__annual_precipitation_mm  \
min                                 0.0                               0.0   
max                                 1.0                               1.0   

     climate__temperature_c_anomaly  \
min                             0.0   
max                             1.0   

     climate__temperature_c_trailing5_slope_per_year  \
min                                              0.0   
max                                              1.0   

     climate__precipitation_mm_anomaly  \
min                                0.0   
max                                1.0   

     climate__precipitation_mm_trailing5_slope_per_year  \
min                                                0.0    
max                                                1.0    

     terrain__mean_elevation_m  terrain__elevation_sd_m  \
min                        0.0                      0.0   
max                        1.0          

## Cell 7 — Save exactly three CSV files

In [23]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for name, part in (("train", train_df), ("validation", val_df), ("test", test_df)):
    output = SPLITS_DIR / f"{name}.csv"
    part.to_csv(output, index=False)
    print(f"Saved {output} ({len(part):,} rows)")

Saved /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/splits/hri/model_v1/train.csv (24,752 rows)
Saved /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/splits/hri/model_v1/validation.csv (5,306 rows)
Saved /Users/keerthikeerthi/Desktop/J26-IT-359/components/habitat_risk/data/splits/hri/model_v1/test.csv (5,292 rows)


## Cell 8 — Verify saved CSVs

In [24]:
for name, expected in (("train", train_df), ("validation", val_df), ("test", test_df)):
    path = SPLITS_DIR / f"{name}.csv"
    check = pd.read_csv(path)
    assert len(check) == len(expected)
    assert list(check.columns) == list(df.columns)
    assert check["dataset_split"].eq(name).all()
    assert check[NORMALIZE_COLS].notna().all().all()
    print(f"{name}.csv: {check.shape}, split verified")
print("Completed. No extra files were generated by this notebook.")

train.csv: (24752, 24), split verified
validation.csv: (5306, 24), split verified
test.csv: (5292, 24), split verified
Completed. No extra files were generated by this notebook.


## Research notes
- Do not include `hri_score_0_100`, `fixed_risk_class`, or `quantile_risk_class` in normalization: they are targets/derived outcomes.
- All nine continuous climate and terrain predictors are transformed. Binary `__was_missing` indicators are deliberately left as 0/1.
- This notebook does **not** regenerate spatial/temporal splits or guarantee that the pre-existing split is spatially independent.
- For production inference, persist the fitted imputer/scaler **outside** `splits/` before ending the session. Refit them only on the same training data if recreating this experiment.
- Exclude target-derived predictors from your model feature list to avoid circular prediction of the HRI formula.